<!-- [c2-01-a] -->
# C2-01 — Retrieval concepts (themes A, B, C)

Re-derived from IBM's *Summarize Private Documents Using RAG, LangChain, and LLMs*
(`clones/IBM-RAG-and-Agentic-AI/02 Build RAG Applications/`). The original stays untouched —
see `C2-analysis.md` for the source-material convention.

**This notebook deliberately does not re-teach the RAG pipeline.** Loading, splitting, embedding,
storing and retrieving were built the hard way in `Demo_C1_project.ipynb`, with a
`ParentDocumentRetriever` on top. Section 0 below hands you that pipeline complete, on purpose.

What's left are the three ideas the original raises and never examines — one per theme:

| theme | the question |
|---|---|
| **A** | You retrieved N chunks. How do they become *one* answer? |
| **B** | "What can't I do in **it**?" — a vector search has no idea what "it" is. |
| **C** | The notebook is called *Summarize private documents*. Can this technique actually summarize a document? |

Corpus is IBM's `companyPolicies.txt` rather than the C1 Maquiavel PDF, for one reason: it's short
enough to read end to end, so when a summary silently drops half the document you can *see* what's
missing. (The Maquiavel corpus comes back later, as the golden set for the eval harness.)

python utils/nbtag.py NOTEBOOK            # dry run, prints the map <br>
python utils/nbtag.py NOTEBOOK --apply    # write/refresh tags.   <br>
python utils/nbtag.py NOTEBOOK --list     # read tags, change nothing


<!-- [setup-a] -->
## Setup

In [1]:
# [setup-b]
import os
import warnings

import wget
from dotenv import load_dotenv

# Legacy chains live in `langchain_classic` under langchain 1.x — see CLAUDE.md.
# They are the subject of study here, not a recommendation.
from langchain_classic.chains import ConversationalRetrievalChain, RetrievalQA
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_core.callbacks import BaseCallbackHandler
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

warnings.filterwarnings("ignore")
load_dotenv()

DATA_DIR = "../../data"
CORPUS = f"{DATA_DIR}/companyPolicies.txt"
URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt"
)

if not os.path.exists(CORPUS):
    wget.download(URL, out=CORPUS)
print("corpus:", CORPUS, os.path.getsize(CORPUS), "bytes")

/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_5834/360917113.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


corpus: ../../data/companyPolicies.txt 15660 bytes


In [2]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

<!-- [0a] -->
## 0. Baseline pipeline — mostly given on purpose

The model, the loader and the splitter below are all things you built in C1. They are handed over
complete so this notebook can be about the three questions, not about re-typing a splitter.

The vector store is **not** handed over, for a reason you'll see in 0.1.

In [3]:
# [0b]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5, max_tokens=512)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

documents = TextLoader(CORPUS).load()
chunks = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0).split_documents(documents)

print(f"{len(chunks)} chunks")

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16 chunks


<!-- [0.1a] -->
### 0.1 — a trap, sprung before you trust any number in this notebook

Themes A and C both end in a *count*: how many LLM calls, how many policies covered. A count is
only worth something if the store underneath it holds what you think it holds.

Run the next cell. It builds the same five documents twice — the way section 0 would have, had it
been handed to you.

In [4]:
# [0.1b] Given: the trap. Read the two numbers before reading the explanation below.
from langchain_core.documents import Document
from langchain_core.embeddings import FakeEmbeddings

_demo_docs = [Document(page_content=f"policy {i}") for i in range(5)]
_demo = Chroma.from_documents(_demo_docs, FakeEmbeddings(size=8), collection_name="c2_trap_demo")
print("after 1st build:", _demo._collection.count())

_demo2 = Chroma.from_documents(_demo_docs, FakeEmbeddings(size=8), collection_name="c2_trap_demo")
print("after 2nd build:", _demo2._collection.count())
print("and the FIRST handle now sees:", _demo._collection.count())

_demo.delete_collection()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


after 1st build: 5
after 2nd build: 10
and the FIRST handle now sees: 10


<!-- [0.1c] -->
Five documents became ten. `from_documents` **appends**; it does not replace. Both handles point at
one collection in a client shared across the whole kernel, so re-running a build cell silently
doubles the corpus — and `k=4` starts returning two distinct chunks plus their copies.

Nothing errors. The notebook just quietly starts lying to you.

**This is the notebook's missing reset**, and the fix is not one rule but two, because these are
different kinds of state:

| | vector store | conversation memory (theme B) |
|---|---|---|
| what it is | expensive derived data | live state of one dialogue |
| **the rule** | **cache** — a second run must change nothing | **reset** — a second run must start from turn zero |
| symptom when the rule is broken | duplicate chunks, corrupted counts, embeddings paid for twice | polluted history, nonsense condensed questions |
| where you enforce it | `build_docsearch` — `[0.2b]` | `fresh_memory` — `[B.2b]` |

Cache the first. Reset the second. Swap them and notebooks become haunted.

**Read the rule row as a specification, not as a description of the cell above it.** Re-running is
an action, not a rule — ⇧⏎ twice, nothing more. It has no correct outcome of its own; the outcome
is whichever one you built. The cell above is deliberately the broken version, which is why
re-running it duplicates instead of doing nothing. **Nothing in this notebook satisfies the rule
yet** — 0.2 is where you make it true, and the assert there is what proves it.

One more distinction, because "failed to cache" has two flavours and `from_documents` picks the
worse one:

- **rebuild, replacing** — the count stays right; you only re-pay for the embeddings.
- **rebuild, appending** — the count goes wrong *and* you re-pay. ← what you just watched happen.

> **This is theme D arriving early.** That table is the two-lifetime problem from the application
> layer: an index cached *per document* versus a conversation reset *per thread* — icebreaker's
> `active_indices[session_id]` and your C1 checkpointer. In a notebook a mix-up costs you a wrong
> number. In a server it costs one user another user's document.


<!-- [0.2a] -->
### 0.2 — build the store so re-running is a no-op

Idempotent here means *build once, reuse after* — not *rebuild deterministically*, which would
re-pay the embedding cost on every run.

One catch to design around: you **will** want to change `chunk_size` or `k` later in this notebook.
A cache that ignores that is worse than no cache, because it hands you stale vectors with a
confident face.

In [36]:
# [0.2b] TODO: build_docsearch(rebuild: bool = False) -> Chroma
# - use a named collection (COLLECTION below) so it can be addressed and deleted deliberately
# - if rebuild is True, drop the existing collection first
# - if it already holds exactly len(chunks) documents, return it as-is without re-embedding
# HINT: Chroma(collection_name=..., embedding_function=embeddings) opens an existing collection
#       without writing; ._collection.count() tells you what's in it; .delete_collection() drops it
# HINT: set rebuild=True yourself whenever you change chunk_size / the splitter

COLLECTION = "c2_01_policies"


def build_docsearch(rebuild: bool = False) -> Chroma:
    """Return a Chroma store over `chunks`, building it only if it isn't already there."""
    vector_store = Chroma(
        collection_name="c2_01_policies",
        embedding_function=embeddings,
    )
    if rebuild :
        vector_store.delete_collection()
        vector_store = Chroma(
            collection_name="c2_01_policies",
            embedding_function=embeddings,
        )
        vector_store.add_documents(chunks)
    if vector_store._collection.count() != len(chunks):
        vector_store.add_documents(chunks)
    return vector_store

In [37]:
# [0.2c] Given: proof it worked. Run this cell twice — the count must not move.
docsearch = build_docsearch()
retriever = docsearch.as_retriever()

assert docsearch._collection.count() == len(chunks), (
    f"expected {len(chunks)} docs, found {docsearch._collection.count()} — "
    "the store is accumulating; re-check build_docsearch"
)
print(
    f"OK — {docsearch._collection.count()} docs, retriever k={retriever.search_kwargs.get('k', 4)}"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


OK — 16 docs, retriever k=4


In [7]:
# [0.2d] Sanity check before building anything on top of it.
retriever.invoke("mobile phone policy")[0].page_content[:300]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


'4.\tMobile Phone Policy'

<!-- [A-a] -->
---

# Theme A — how do N chunks become one answer?

The retriever hands back k chunks. The LLM produces one answer. **The step in between has a name,
and it has options.**

The original passes `chain_type="stuff"` without comment. Three alternatives exist:

| chain_type | what it does | LLM calls |
|---|---|---|
| `stuff` | concatenate all chunks into one prompt | 1 |
| `map_reduce` | answer from each chunk separately, then combine the answers | N + 1 |
| `refine` | answer from chunk 1, then revise it chunk by chunk | N |

They differ in cost, in latency, and in what they do when the chunks disagree or don't fit in the
context window. Right now you have no basis for choosing between them — that's the point of the
exercise.

<!-- [A.1a] -->
### A.1 — a factory, so the three are comparable

Write one function that builds a `RetrievalQA` for a given `chain_type`, so the only thing varying
between the three runs is the strategy.

In [27]:
# [A.1b] TODO: build_qa(chain_type: str) -> RetrievalQA
# - use RetrievalQA.from_chain_type
# - llm=llm, retriever=retriever
# - return_source_documents=True  (theme C needs it later, and it costs nothing now)


def build_qa(chain_type: str) -> RetrievalQA:
    """Build a RetrievalQA over `retriever` using the given chain_type."""
    qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True, chain_type=chain_type)
    return qa

In [28]:
# [A.1c] TODO: test it standalone before comparing anything — build one, invoke it, look at the keys
# of what comes back. What does return_source_documents add to the result dict?
query_1 = "mobile phone policy"

qa = build_qa("stuff")
qa.invoke(query_1)

{'query': 'mobile phone policy',
 'result': "The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization. Here are the key points:\n\n1. **Acceptable Use**: Mobile devices should primarily be used for work-related tasks. Limited personal usage is permitted as long as it does not disrupt work obligations.\n\n2. **Security**: Employees must safeguard their mobile devices and access credentials, exercise caution when downloading apps or clicking links from unfamiliar sources, and promptly report any security concerns or suspicious activities.\n\n3. **Confidentiality**: Sensitive company information should not be transmitted via unsecured messaging apps or emails, and employees should be discreet when discussing company matters in public spaces.\n\n4. **Cost Management**: Employees should keep personal phone usage separate from company accounts and reimburse the company for any personal charges incurr

<!-- [A.2a] -->
### A.2 — run the same question three ways

Pick a question the document genuinely answers, and send it through all three strategies.

In [29]:
# [A.2b] TODO: for each of "stuff", "map_reduce", "refine":
#   - build the chain, invoke it with the SAME question
#   - print the chain_type and the answer
# Read the three answers side by side before moving on. Are they different? How?

question = "What is the mobile phone policy?"

In [30]:
qa_stuff = build_qa("stuff")
stuff = qa_stuff.invoke(question)


In [31]:
qa_map_reduce = build_qa("map_reduce")
map_reduce = qa_map_reduce.invoke(question)

In [32]:
qa_refine = build_qa("refine")
refine = qa_refine.invoke(question)

In [33]:
stuff["result"][:200]
map_reduce["result"][:200]
refine["result"][:200]

'The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization. Its purpose is to ensure that employees use mobil'

'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. Key points of the policy include:\n\n- **Acceptab'

'The original answer regarding the Mobile Phone Policy remains relevant and comprehensive, and the context about the Smoking Policy does not seem to provide additional insights or require any modificat'

<!-- [A.3a] -->
### A.3 — now the part that makes it a decision

Three similar answers tell you nothing about which to use. **Cost does.** Count the LLM calls each
strategy actually makes.

A `BaseCallbackHandler` counting `on_llm_start` is the cheapest way to see it.

In [34]:
# [A.3b] TODO: define a callback handler that counts LLM starts
# HINT: subclass BaseCallbackHandler, override on_llm_start(self, serialized, prompts, **kwargs),
#       and increment a counter (remember: one call can carry several prompts)


class CallCounter(BaseCallbackHandler):
    """Counts how many times the LLM is invoked during a chain run."""

    def __init__(self) -> None:
        self.starts = 0  # on_llm_start events
        self.prompts = 0  # prompts carried across those events

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.starts += 1
        self.prompts += len(prompts)


In [35]:
# [A.3c] TODO: re-run the three strategies, this time passing config={"callbacks": [counter]}
# Print a small table: chain_type | n_llm_calls | len(answer)

rows = []
for chain_type in ("stuff", "map_reduce", "refine"):
    counter = CallCounter()
    result = build_qa(chain_type).invoke(question, config={"callbacks": [counter]})
    rows.append((chain_type, counter.starts, counter.prompts, len(result["result"])))

print(f"{'chain_type':<12}{'starts':>8}{'prompts':>9}{'len(answer)':>13}")
for chain_type, starts, prompts, n in rows:
    print(f"{chain_type:<12}{starts:>8}{prompts:>9}{n:>13}")


chain_type    starts  prompts  len(answer)
stuff              1        1         1775
map_reduce         5        5         1676
refine             4        4          405


<!-- [A-what-a] -->
### What to take away

> Does the call count match the table above (1 / N+1 / N)?
> If `stuff` gives a comparable answer for 1 call instead of 5, when would you ever pay for the
> others? Write your answer down before moving on — theme C is where it stops being hypothetical.

---

# Theme B — the follow-up problem

Two turns. The second one only makes sense if you remember the first.

In [17]:
# [B-a] Given: the trap. Run it and read the second answer carefully.
qa = build_qa("stuff")

turn_1 = qa.invoke("What is the mobile phone policy?")
print("Q1:", turn_1["result"][:400], "\n")

turn_2 = qa.invoke("What can't I do in it?")
print("Q2:", turn_2["result"][:400])

Q1: The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. Its key points include:

- **Acceptable Use**: Mobile devices are primarily for work-related tasks, with limited personal usage allowed as long as it doesn't disrupt work obligations.
- **Security**: Employees should safeguard their mobile devices 

Q2: In the context of the Internet and Email Policy, you cannot:

1. Use company-provided internet and email services for non-job-related tasks during work hours.
2. Share your login credentials or passwords with others.
3. Open email attachments or links from unknown sources without caution.
4. Transmit confidential information or sensitive data via email without encryption.
5. Engage in harassment, 


<!-- [B.1a] -->
### B.1 — diagnose before fixing

The second answer is wrong, or vague, or about the wrong policy. **Don't accept the obvious
explanation without checking it.** The claim is: the retriever never saw the word "mobile", so it
retrieved on "what can't I do", which matches half the document.

Prove it.

In [39]:
# [B.1b] TODO: retrieve directly for the follow-up question (no chain, just retriever.invoke)
# and print the first ~200 chars of each returned chunk.
# Which policies came back? Is "mobile" among them?

docsearch = build_docsearch()
retriever = docsearch.as_retriever()
result = retriever.invoke("What can't I do in it?")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [44]:
len(result)
result[0].page_content[:200]
result[1].page_content[:200]
result[2].page_content[:200]
result[3].page_content[:200]

4

'Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and '

'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that em'

'Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environm'

'Policy Objective: The Drug and Alcohol Policy is established to establish clear expectations and guidelines for the responsible use of drugs and alcohol within the organization. This policy aims to ma'

<!-- [B.2a] -->
### B.2 — the fix, and the thing worth looking at

`ConversationalRetrievalChain` inserts a step before retrieval: it sends the chat history plus the
new question to the LLM and asks for a **standalone question** — "What can't I do in it?" becomes
something like "What are the restrictions in the company mobile phone policy?"

That rewritten question is the whole mechanism, and it's normally invisible.
`return_generated_question=True` exposes it.

Per 0.1, memory is the state that must **reset**, not cache. Every experiment below starts from
turn zero — otherwise the condensed question is computed against a history containing your previous
three attempts, and you'll be debugging your own leftovers.

In [19]:
# [B.2b]

# TODO: fresh_memory() -> ConversationBufferMemory
# - memory_key="chat_history", return_messages=True
# - a NEW instance every call — that is the entire point
# HINT: the kwarg is return_messageS. The original IBM notebook writes return_message,
#       which silently does nothing.
def fresh_memory() -> ConversationBufferMemory:
    """A conversation memory with no history. Call at the start of every experiment."""
    return ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer" )


# TODO: build_conv_qa() -> ConversationalRetrievalChain
def build_conv_qa() -> ConversationalRetrievalChain:
    return ConversationalRetrievalChain.from_llm(
        llm=llm, retriever=retriever, memory=fresh_memory(), return_generated_question=True
    )


# - building it should imply a fresh conversation, so call fresh_memory() inside

In [20]:
# [B.2c] TODO: run the same two turns through a freshly built chain.
# For each turn print BOTH result["answer"] and result["generated_question"].
# Turn 1's generated question should be boring. Turn 2's is the interesting one.

# Then, to prove the reset matters: run this whole cell a second time. The generated question for
# turn 2 should be IDENTICAL to the first run. If it drifts, memory is leaking between runs.


conv_qa = build_conv_qa()
questions = ["What is the mobile phone policy?", "What can't I do in it?"]
for i in questions:
    result = conv_qa.invoke({"question": i})
    print(result["answer"])
    result["generated_question"]    


The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization. The key points of the policy include:

1. **Acceptable Use**: Mobile devices are primarily for work-related tasks, with limited personal usage allowed as long as it does not disrupt work obligations.

2. **Security**: Employees must safeguard their mobile devices and access credentials, exercise caution when downloading apps or clicking links from unfamiliar sources, and report any security concerns or suspicious activities promptly.

3. **Confidentiality**: Employees should avoid transmitting sensitive company information via unsecured messaging apps or emails and be discreet when discussing company matters in public.

4. **Cost Management**: Personal phone usage should be kept separate from company accounts, and employees must reimburse the company for any personal charges on company-issued phones.

5. **Compliance**: Adherence to all 

'What is the mobile phone policy?'

The following actions are prohibited under the Mobile Phone Policy:

1. Transmitting sensitive company information via unsecured messaging apps or emails.
2. Disrupting work obligations with excessive personal mobile phone usage.
3. Failing to safeguard mobile devices and access credentials.
4. Downloading apps or clicking links from unfamiliar sources without caution.
5. Not reporting lost or stolen mobile devices immediately to the IT department or supervisor.
6. Mixing personal phone usage with company accounts and not reimbursing the company for personal charges on company-issued phones.
7. Non-compliance with relevant laws and regulations concerning mobile phone usage.

Non-compliance with the policy may also lead to disciplinary actions, including the potential loss of mobile phone privileges.


'What actions are prohibited under the mobile phone policy?'

<!-- [B.3a] -->
### B.3 — the comparison that only you can make

C1 solved this same problem a completely different way: no condensation step at all. The
checkpointer replayed the conversation, and the *agent* decided what to pass to `get_context`.

> Two mechanisms, same symptom cured:
> - condensation — one extra LLM call, always, whether or not the question needs it
> - agent reasoning — no extra call, but the query is chosen by a model that could choose badly
>
> Which one fails more gracefully? Which would you rather debug at 2am?
> When does the condensation step actively *hurt* — think about a follow-up that isn't a
> follow-up at all.

---

# Theme C — can this thing actually summarize?

The original notebook is titled *Summarize Private Documents* and asks
*"Can you summarize the document for me?"* through a `k=4` retriever.

Before running it: **establish ground truth.** You can read this corpus.

In [21]:
conv_qa = build_conv_qa()
questions = ["What are the 9 company policies in this document?"]
for i in questions:
    result = conv_qa.invoke({"question": i})
    print(result["answer"])
    result["generated_question"]  

I don't know.


'What are the 9 company policies in this document?'

In [22]:
# [C-a] TODO: how many distinct policies are in companyPolicies.txt?
# Read the raw file (or skim `documents[0].page_content`) and list their names.
# This is the answer key for everything below.
# print(documents[0].page_content)

# Code of Conduct
# Recruitment Policy
# Internet and Email Policy
# Mobile Phone Policy
# Smoking Policy
# Drug and Alcohol Policy
# Health and Safety Policy
# Anti-discrimination and Harassment Policy
# Discipline and Termination Policy


<!-- [C.1a] -->
### C.1 — ask for a summary through retrieval

In [23]:
# [C.1b] TODO: ask build_qa("stuff") to summarize the document.
# Then: how many of the policies you listed above appear in the answer?
# And separately — how many chunks did it actually see? (result["source_documents"])

qa = build_qa("stuff")

result = qa.invoke("summarize the document")

In [24]:
result["result"]
len(result["source_documents"])
result["source_documents"][0].page_content[:150]
result["source_documents"][1].page_content[:150]
result["source_documents"][2].page_content[:150]
result["source_documents"][3].page_content[:150]

"The document outlines several key policies within an organization, including:\n\n1. **Health and Safety Policy**: Emphasizes the importance of maintaining a safe workplace, complying with health and safety regulations, and ensuring the well-being of employees and the public. It encourages individual responsibility, regular assessments, training, and open communication regarding safety.\n\n2. **Discipline and Termination Policy**: Details the organization's commitment to a productive and respectful work environment. It sets expectations for employee performance and conduct, outlines potential disciplinary actions (such as warnings and suspension), and describes the termination process for persistent issues or policy violations. The policy ensures fairness, legal compliance, and a smooth exit process for departing employees.\n\nOverall, the document serves as a framework for maintaining safety, discipline, and ethical standards within the organization."

4

'9.\tDiscipline and Termination Policy'

'7.\tHealth and Safety Policy\n\nOur commitment to health and safety is paramount. We prioritize the well-being of our employees, customers, and the publi'

"The Discipline and Termination Policy underscores the organization's commitment to maintaining a productive, ethical, and respectful work environment."

'5.\tSmoking Policy'

<!-- [C.2a] -->
### C.2 — name the gap

> The retriever returned k chunks out of the total. A summary of k chunks is not a summary of the
> document — it's a summary of whatever happened to embed near the word "summarize".
>
> Notice this is not a prompt problem. No amount of prompt engineering makes the model summarize
> text it was never given. **Retrieval answers questions; summarization needs the whole corpus.**
> They are different operations that happen to share a pipeline.

<!-- [C.3a] -->
### C.3 — do it properly

Drop the retriever entirely. `load_summarize_chain` runs over *all* chunks — and now
`map_reduce` / `refine` from theme A stop being an academic choice, because "stuff the whole
document into one prompt" is exactly what stops working as documents grow.

In [ ]:
# [C.3b] TODO: summarize ALL chunks with load_summarize_chain(llm, chain_type="map_reduce")
# - it takes documents directly: chain.invoke({"input_documents": chunks})
# - count the policies in this answer, against the same answer key
# - count the LLM calls with your CallCounter

counter = CallCounter()
summarize = load_summarize_chain(
    llm=llm,
    chain_type="map_reduce"
)
result = summarize.invoke({"input_documents": chunks}, config={"callbacks": [counter]})
print(result["output_text"])

rows = []
rows.append(("map_reduce", counter.starts, counter.prompts, len(result["output_text"])))
print(rows)

The document outlines several key organizational policies that promote ethical behavior, a positive work environment, and compliance with laws. 

1. **Code of Conduct**: Establishes principles of integrity, respect, accountability, and safety, fostering an inclusive workplace and guiding ethical decision-making.

2. **Recruitment Policy**: Details fair and equitable hiring practices, emphasizing diversity, transparency, and adherence to legal standards to build a talented workforce.

3. **Internet and Email Policy**: Sets guidelines for responsible use of digital communication tools, focusing on work-related use, security, and compliance with legal standards.

4. **Mobile Phone Policy**: Outlines acceptable mobile phone usage during work hours to minimize distractions and maintain professionalism.

5. **Smoking Policy**: Promotes a smoke-free environment by restricting smoking to designated areas and ensuring compliance with health regulations.

6. **Drug and Alcohol Policy**: Prohibit

In [ ]:
# [C.3c] TODO: same thing with chain_type="refine". Compare all three summaries:
#   retrieval-based | map_reduce | refine
# on two axes: coverage (policies mentioned / total) and cost (LLM calls).
counter = CallCounter()
summarize = load_summarize_chain(
    llm=llm,
    chain_type="refine"
)
result = summarize.invoke({"input_documents": chunks}, config={"callbacks": [counter]})
print(result["output_text"])

rows = []
rows.append(("refine", counter.starts, counter.prompts, len(result["output_text"])))
print(rows)

The "Code of Conduct" outlines the fundamental principles and ethical standards that guide every member of our organization, emphasizing integrity, respect, accountability, safety, and environmental responsibility. It serves as a framework for maintaining a positive and professional work environment by encouraging honest and transparent interactions, fostering an inclusive culture, and holding individuals accountable for their actions. The Code is not merely a set of rules but the foundation of our organizational culture, with an expectation that all employees uphold these principles and act as role models, thereby ensuring our commitment to ethical conduct and social responsibility.

In conjunction with our Recruitment Policy, the Code of Conduct is integral to attracting and selecting individuals who align with our values, ensuring that new hires contribute to a culture of integrity and respect from the outset. Our Recruitment Policy reflects our commitment to attracting, selecting, 

<!-- [C-what-a] -->
### What this sets up

> You now have a concrete case where `stuff` is not merely cheaper but *wrong*, and where the
> theme A choice has a measurable right answer. Coverage is a crude metric — but it is a metric,
> which is more than "the answer reads fine" gives you. That's the seed of the eval harness
> (`C2-analysis.md`, cross-cutting section).

LlamaIndex names this distinction in its API — a `SummaryIndex` traverses everything while a
`VectorStoreIndex` retrieves, and `tree_summarize` is `map_reduce` under a different name. That
comparison is theme E, in `C2-02`.

---

## Where this lands

- **Themes A, B, C** — done here, in a notebook, because they're ideas that need values on screen.
- **Theme D** (`src/ibm_rag_and_agentic_ai/c2/`) — the application layer: entrypoints, runtime
  ingestion, session state. Not teachable in a notebook, which is the whole reason it's a module.
- **Open loop from C1** — `quoted_excerpts` / source attribution. `return_source_documents=True`
  has been on every chain in this notebook; that's the raw material for closing it.